# Label-free self-supervised stereo depth

Re-implementation of the stereo architecture from *A Learned Stereo Depth System for
Robotic Manipulation in Homes* (Shankar et al., arXiv:2109.11644), trained **without any
ground-truth disparity or depth**.

### The one rule

| Mode | Ground truth |
|---|---|
| `train_unlabeled` | **never read** |
| `adapt_unlabeled` | **never read** |
| `evaluate` | read, for metrics only, on a frozen checkpoint |

Every ground-truth cell is guarded by `if MODE == "evaluate"` and refuses to run otherwise.

### How to use this notebook

1. **Edit the Control Panel (cell 3) — and nothing else.** Every knob lives there; the rest
   of the notebook derives from it. Adding a dataset, changing resolution or switching to
   evaluation are all one-line edits in that single cell.
2. **Run all.** The preflight cell right after it prints exactly what will happen and warns
   about anything inconsistent *before* any training starts.
3. Settings -> Accelerator -> **GPU T4 x2** or **P100**, and Internet **on** if you want the
   notebook to download datasets itself.

## What the paper evaluates on — and why this notebook holds it out

The paper reports accuracy on exactly two benchmarks:

| Paper | Evaluation set | Published result | Reproducible here? |
|---|---|---|---|
| Table IV | Scene Flow FlyingThings3D **TEST** split | EPE **0.936**, %Bad(1.0) **10.0** | **Yes**, if your FlyingThings3D mirror ships the TEST split with disparity |
| Table V | Middlebury 2014 **TEST** split | bad2.0 12.7/17.4, avgerr 3.6/5.77 | **No — not by anyone.** The ground truth is held privately by the Middlebury evaluation server; those numbers exist only through online submission |

So training here uses **train splits only**:

* **FlyingThings3D** — `SCENEFLOW_SPLIT = "TRAIN"`. The TEST split is the paper's Table IV
  benchmark and is never trained on, which is what makes a comparison against 0.936 / 10.0
  meaningful. The preflight warns if you change this.
* **KITTI (Eigen split)** — the paper reports **no KITTI accuracy at all**, only runtimes
  (Table III), so there is no KITTI evaluation set to protect. It is pure extra real-world
  imagery for the photometric objective. Note this is a deviation: the paper does not train
  on KITTI.
* **Middlebury** — the paper *trains* on the Middlebury training set (repeated 100x per
  epoch) and evaluates on the hidden test set. Training on it here therefore follows the
  paper. The consequence: since the Middlebury **training** split is the only one with public
  ground truth, using it for training means it **cannot** also serve as a local benchmark.
  `PROTOCOL = "middlebury2014"` would then be scoring the model on its own training images.

**Recommended benchmark: `PROTOCOL = "sceneflow"`** with `SCENEFLOW_SPLIT` left at `"TRAIN"`
for training — evaluation switches to the TEST split automatically. That is the one row that
can be compared directly against a published number.

If you would rather benchmark on Middlebury instead, set `DATASETS["middlebury"] = 0` so it is
excluded from training; it then becomes a clean held-out set, though still a different split
from the paper's Table V.

## 1. Configuration guide

What every knob in the Control Panel means. **You do not need to read this to start** —
the defaults train on Middlebury and evaluate with the paper's Middlebury metric set.

### Where the code comes from
| Setting | Meaning |
|---|---|
| `GITHUB_USER` | Your GitHub username. Leave `""` to load the code from an attached Kaggle Dataset instead (no GitHub account needed). |
| `GITHUB_REPO` | Repository name, if you pushed one. |

### What to run
| Setting | Meaning |
|---|---|
| `MODE` | `train_unlabeled` = Stages 1+2 from random init. `adapt_unlabeled` = Stage 3, fine-tune an existing checkpoint on your own camera. `evaluate` = Stage 4, freeze a checkpoint and measure it against ground truth. |

### Which data
| Setting | Meaning |
|---|---|
| `DATASETS` | `name -> sampling weight`. The weight is the **share of training draws, not a share of images**: a 15-scene set at 0.5 is drawn as often as a 20 000-frame set at 0.5. Set a weight to `0` to exclude it — only non-zero entries are downloaded. This one dict drives downloading, previewing and training. |
| `ATTACHED` | `name -> /kaggle/input/...` for datasets you **attach** instead of downloading. Anything listed here is read straight from that path, so Kaggle's 20 GB working-directory quota never applies. **This is how to use Scene Flow on Kaggle.** The mirror's internal folder layout does not matter — see below. |
| `CUSTOM_DATA` | Your own capture for `adapt_unlabeled`: a folder with `left/` and `right/` and no labels. |
| `DATASET_ROOT` | Where *downloaded* datasets go. Ignored for anything in `ATTACHED`. |

#### Using Scene Flow (or any large dataset) without downloading it

Scene Flow is 132 GB officially — far beyond Kaggle's quota. Attach a community mirror instead:

1. **Add Data** → search `sceneflow` → attach it.
2. Note the path it appears at under `/kaggle/input/`.
3. In the Control Panel:

```python
DATASETS = {"sceneflow": 1.0}                       # give it a weight
ATTACHED = {"sceneflow": "/kaggle/input/sceneflow"} # and point at it
```

Nothing is downloaded for it. **The mirror's internal layout does not matter** — the loader
searches the attached directory (up to 5 levels deep) for either structure:

| Layout | Looks like |
|---|---|
| official | `frames_finalpass/TRAIN/<A\|B\|C>/<scene>/left/*.png` + `disparity/TRAIN/...` |
| subset | `train/image_clean/left/*.png` + `train/disparity/left/*.pfm` |

It falls back to `frames_cleanpass` if there is no `frames_finalpass`, and maps `TEST` onto
the subset release's `val` directory. The "Attached datasets" cell prints exactly what it
resolved, and if a mirror is not recognised it prints that mirror's real directory tree
rather than failing opaquely.

A mirror with **images but no disparity is still fully usable for training** — training here
is label-free. Disparity is only needed for Stage 4 benchmarking, and the loader says so
explicitly if you try to benchmark against an images-only copy.

### Resolution and disparity range
| Setting | Meaning |
|---|---|
| `TRAIN_WIDTH`, `TRAIN_HEIGHT` | Every sample is **resized** to this (never cropped), so batches are rectangular. Width also sets the disparity search range: `num_disparities = a fixed range at a canonical width`. |
| `DOWNSAMPLE` | Cost volume resolution: `4` = 1/4 (the paper's low-res variant), `8` = 1/8 (high-res variant, cheaper). |
| `DISPARITY_RANGE` | How far the model searches, in `TRAIN_WIDTH` pixels. `"auto"` measures it from your images with no labels. Smaller is more accurate as long as it covers your scenes — measured on real data, 64 gives 6.2px error where 320 gives 10.8px. Part of the weights, so changing it starts a new model. |

**The number that matters is your disparity range, not your image size.** `d_max = f·B/Z_min`
at your native width, scaled by `TRAIN_WIDTH / NATIVE_WIDTH`. The preflight cell checks this
against your data and warns if the model cannot reach far enough.

### Optimisation
| Setting | Meaning |
|---|---|
| `EPOCHS`, `BATCH_SIZE`, `LEARNING_RATE` | Standard. Batch size is bounded by the cost volume, which grows with `TRAIN_WIDTH × TRAIN_HEIGHT × num_disparities / DOWNSAMPLE³` — the preflight prints the per-image cost. |
| `NUM_WORKERS` | Dataloader processes. `2` suits Kaggle. |

### Loss weights — the label-free objective
`L = W_PHOTOMETRIC·L_photo + W_SMOOTHNESS·L_smooth + W_LEFT_RIGHT·L_lr + W_LOW_RESOLUTION·(photo+smooth at cost-volume scale) + W_PSEUDO·ramp·L_pseudo + W_CONFIDENCE·L_conf`

| Setting | Meaning |
|---|---|
| `W_PHOTOMETRIC` | SSIM+L1 reconstruction — the primary signal. Leave at 1.0 and scale the others relative to it. |
| `W_SMOOTHNESS` | Edge-aware smoothness. Too high collapses disparity toward a constant; too low leaves textureless regions noisy. |
| `W_LEFT_RIGHT` | Geometric agreement between the two disparity maps. Also what handles occlusions. |
| `W_LOW_RESOLUTION` | Applies photometric+smoothness to the soft-argmin output too. Without it the cost volume gets gradient only through the refinement net, which learns to ignore a bad coarse input rather than fix it. |
| `W_PSEUDO` | Teacher pseudo-label consistency (Stage 2). |
| `W_CONFIDENCE` | Label-free matchability target. **This one is my addition, not the paper's, and is unvalidated** — set to `0.0` to disable it while keeping the architecture intact. |

### Teacher (Stage 2 self-training)
| Setting | Meaning |
|---|---|
| `| `TEACHER_EMA_DECAY` | `θ_teacher ← m·θ_teacher + (1−m)·θ_student`. Higher = slower, more stable teacher. |
| `FILTER_CONFIDENCE`, `FILTER_LR_PIXELS`, `FILTER_PHOTOMETRIC` | A teacher pixel becomes a pseudo-label only if it passes **all** of these. Watch `pseudo_cov` in the training log: near 0 means the filter rejects everything, near 1 means it copies the teacher's errors wholesale. |

### Evaluation (Stage 4)
| Setting | Meaning |
|---|---|
| `PROTOCOL` | Which benchmark protocol. `EVAL_ROOT` is **derived from this** — no second path to keep in sync. |
| `CHECKPOINT` | `None` = use `best.pt` from this notebook's training run. |
| `POSTPROCESS_CONFIDENCE`, `POSTPROCESS_MIN_REGION` | The paper's on-robot filter (`exp(matchability) ≥ 0.25`, region ≥ 2000 px). The paper's **tables use raw output**, so this is always reported as a separate row. |

---

### Common recipes

<details><summary><b>Train on Middlebury only (the default)</b></summary>

```python
MODE = "train_unlabeled"
DATASETS = {"middlebury": 1.0}
```
</details>

<details><summary><b>Add KITTI to the training mixture</b></summary>

```python
DATASETS = {"middlebury": 0.5, "kitti2015": 0.3, "kitti2012": 0.2}
```
One edit. Downloading, previewing and the training mixture all follow automatically.
</details>

<details><summary><b>Use Scene Flow from an attached Kaggle dataset (not downloaded)</b></summary>

```python
DATASETS = {"sceneflow": 1.0}
ATTACHED = {"sceneflow": "/kaggle/input/sceneflow"}   # whatever path Add Data gave you
PROTOCOL = "sceneflow"                                # for MODE="evaluate"
```
Two lines. Nothing is downloaded, and the mirror's internal folder layout is discovered
automatically.
</details>

<details><summary><b>Train on your own unlabeled camera</b></summary>

```python
MODE        = "adapt_unlabeled"
CUSTOM_DATA = "/kaggle/input/my-stereo-camera"   # contains left/ and right/
INIT_CHECKPOINT = "/kaggle/working/outputs/train_unlabeled/best.pt"
```
Adaptation starts from an existing checkpoint at a reduced learning rate (`ADAPT`).
</details>

<details><summary><b>Small images, e.g. 224x224</b></summary>

```python
TRAIN_HEIGHT = 224
TRAIN_WIDTH  = 224      # -> num_disparities = 112, max disparity 107 px
BATCH_SIZE   = 24       # 6 MB/image instead of 79 at 640x384
```
Do **not** upscale small images to 640x384: it multiplies disparity by 2.9, adds no
information, and costs 13x the cost-volume memory. And do not squash a wide capture into a
square — horizontal resolution *is* depth precision (`δZ = Z²/(fB)·δd`, and halving the
width halves `f`).
</details>

<details><summary><b>Evaluate a checkpoint against ground truth</b></summary>

```python
MODE       = "evaluate"
PROTOCOL   = "middlebury2014"
CHECKPOINT = "/kaggle/working/outputs/train_unlabeled/best.pt"
```
</details>

## 2. Control Panel

**This is the only cell you need to edit.** Everything below derives from it.

In [ ]:
# =============================================================================
#  CONTROL PANEL
# =============================================================================

MODE = "train_unlabeled"   # train_unlabeled | adapt_unlabeled | evaluate

# --- Data --------------------------------------------------------------------
# name -> share of training draws (not of images: a 23-scene set at 0.25 is
# drawn as often as a 20000-frame one at 0.25). 0 excludes a dataset.
DATASETS = {"sceneflow": 0.50, "kitti": 0.25, "middlebury": 0.25}

MAX_TRAIN_SAMPLES = None   # cap total training pairs (None = all). Split across
                           # the datasets above in proportion to their weights.
MAX_EVAL_SAMPLES  = None   # cap benchmark images (None = all)

# --- Training ----------------------------------------------------------------
EPOCHS        = 100
BATCH_SIZE    = 8
TRAIN_WIDTH   = 640        # the ONE fixed dimension. Heights follow each image's
                           # own aspect ratio, so nothing is squashed or cropped.
                           # The model runs at any resolution afterwards.
LEARNING_RATE = 1e-4

DISPARITY_RANGE = 128      # how far the model searches, in TRAIN_WIDTH pixels.
                           # 128 = 20% of width: the smallest range that covers
                           # all three training sets without clipping, and 2.2x
                           # faster and 36% smaller than the old 320. Set "auto"
                           # to measure it from your own images instead (cell 5b).
                           # Baked into the weights: changing it starts a new model.

# --- Evaluation (MODE = "evaluate") ------------------------------------------
PROTOCOL   = "sceneflow"   # sceneflow | middlebury2014 | eth3d | kitti2015 | kitti2012
CHECKPOINT = None          # None -> best.pt from the training run

# --- Continuing a previous session -------------------------------------------
# The last cell writes <output>/checkpoints.zip. Upload it to Google Drive,
# share it "Anyone with the link", and paste the link here to carry on.
# A local path or an attached Kaggle Dataset works too.
RESUME_ARCHIVE = ""

# =============================================================================
print(f"MODE = {MODE}   datasets = {[k for k, v in DATASETS.items() if v > 0]}")
print("Everything else lives in the Advanced cell below; the preflight reports "
      "what it all resolves to.")


### Advanced settings

Defaults that rarely need touching: where the code and data come from, the loss
weights, the teacher, augmentation and visualisation. **Skip this cell unless you
are changing the method itself** — the preflight below prints what everything
resolved to either way.

In [ ]:
# --- Where the code comes from -----------------------------------------------
GITHUB_USER, GITHUB_REPO, GITHUB_BRANCH = "hnquang-cs", "stereo-depth", "main"
REFRESH_CODE = True        # Kaggle persists /kaggle/working, so a stale checkout
                           # would otherwise shadow your newest commits

# --- Where the data is -------------------------------------------------------
# Attached as Kaggle inputs; nothing is downloaded. The folder layout inside each
# mirror does not matter -- every loader searches it for the pair of views.
ATTACHED = {
    "sceneflow":  "/kaggle/input/datasets/kiraarsene/flying-things-3d",
    "kitti":      "/kaggle/input/datasets/hocop1/kitti-odometry",
    "middlebury": "/kaggle/input/datasets/minhanhtruong/middleburystereodataset",
}
DATASET_ROOT = "/kaggle/working/datasets"
CUSTOM_DATA  = "/kaggle/input/my-stereo-camera"    # adapt_unlabeled: left/ and right/

SCENEFLOW_SPLIT  = "TRAIN"   # the val keys are the paper's benchmark; keep them out
SCENEFLOW_SUBSET = None      # None | FlyingThings3D | Driving | Monkaa
SCENEFLOW_PASS   = None      # None | finalpass | cleanpass
KITTI_VERSION    = None      # None auto-detects (raw | odometry | 2015 | 2012)

# --- Model -------------------------------------------------------------------
PRESERVE_ASPECT   = True     # fix the width, let the height follow each image's
                             # ratio. Inference already does this, so turning it
                             # off re-introduces a train/test geometry mismatch.
TRAIN_HEIGHT      = 384      # only used when PRESERVE_ASPECT is False

DOWNSAMPLE        = 4        # cost volume resolution: 4 = 1/4, 8 = 1/8
RESIDUAL_LIMIT    = None     # bound the refinement residual, as a fraction of the
                             # search range. None = the reference unbounded head.
SOFT_ARGMIN_WINDOW = None    # restrict the disparity expectation to +/- N bins around
                             # the cost minimum. A big win on the photometric cost
                             # volume (9.14 -> 6.84 px), but it did NOT replicate on a
                             # trained network's own volume, where every read-out
                             # scores the same and None scored best. Left off.

# DISPARITY_RANGE = "auto" calibration. Label-free: reads the two views only.
CALIB_PAIRS      = 24        # pairs to measure; a p99 over millions of pixels converges fast
CALIB_PERCENTILE = 99.0      # ignore the top 1%, which is dominated by mismatches
CALIB_MARGIN     = 1.15      # safety factor on the measured percentile

# --- Loss weights ------------------------------------------------------------
# The Monodepth objective (Godard et al. 2017, eq. 2): photometric + left-right
# + smoothness, and nothing else. Their a_ap = 1, a_lr = 1, a_ds = 0.1.
#
# a_lr = 1 is only meaningful because Monodepth's disparity is a FRACTION of
# image width. The disparity-space terms here are normalised by the search range
# for that reason, so 1.0 means what it means in the paper.
W_PHOTOMETRIC    = 1.0    # a_ap -- SSIM+L1 reconstruction (0.85 / 0.15 internally)
W_LEFT_RIGHT     = 1.0    # a_lr -- geometric agreement between the two disparity maps
W_SMOOTHNESS     = 0.1    # a_ds -- edge-aware

# Below: not part of Monodepth. cost_volume is kept because this architecture
# has a cost volume and Monodepth does not -- see its note. The rest are off.
W_LOW_RESOLUTION = 0.0    # closest analogue of Monodepth's 4-scale sum; set 1.0
                          # for a closer match to the paper
W_CONFIDENCE     = 0.0    # label-free matchability target
W_RANGE_PENALTY  = 0.0    # keeps predictions inside the search range


# --- Stage 3 adaptation ------------------------------------------------------
INIT_CHECKPOINT = None
ADAPT = dict(lr_scale=0.1)

# --- Everything else ---------------------------------------------------------
NUM_WORKERS = 4              # loader processes. Colour jitter is the dominant
                             # per-sample cost (~13-26 ms/pair), so too few workers
                             # starves the GPU: it bursts, then idles while the CPU
                             # catches up. 4 suits a Kaggle T4 x2.
RESUME_WHICH = "last"            # "last" continues; "best" restarts from best weights
VISUALIZE_EVERY, VISUALIZE_SAMPLES = 5, 2
POSTPROCESS_CONFIDENCE, POSTPROCESS_MIN_REGION = 0.25, 2000
print("advanced settings loaded")


## 3. Get the code

Two ways, and the cell picks whichever is available: an attached Kaggle Dataset (no GitHub
account, works offline) or a `git clone` if you set `GITHUB_USER`.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_DIR = "/kaggle/working/stereo-depth"
REPO_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git" if GITHUB_USER else None

#: Modules this notebook imports. If a checkout lacks one it is stale, and the
#: failure needs to say so rather than surface later as a ModuleNotFoundError
#: from somewhere deep in the run.
REQUIRED_MODULES = [
    "stereo/model/stereo_net.py",
    "stereo/data/discovery.py",
    "stereo/data/hdf5_stereo.py",
    "stereo/training/loop.py",
]


def _looks_like_this_repo(directory):
    return (os.path.isfile(os.path.join(directory, "train.py"))
            and os.path.isfile(os.path.join(directory, "stereo", "model", "stereo_net.py")))


def _find_attached_dataset():
    """Search every attached Kaggle input, regardless of the dataset's slug."""
    if not os.path.isdir("/kaggle/input"):
        return None
    for candidate in sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*")):
        if os.path.isdir(candidate) and _looks_like_this_repo(candidate):
            return candidate
    return None


def _missing_modules(directory):
    return [m for m in REQUIRED_MODULES if not os.path.isfile(os.path.join(directory, m))]


# Kaggle persists /kaggle/working, so an old checkout silently shadows new commits.
if REFRESH_CODE and os.path.isdir(REPO_DIR):
    print(f"removing the previous checkout at {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

source_used = None
if not os.path.exists(REPO_DIR):
    # Prefer git: a clone is always current, whereas an attached Kaggle Dataset is
    # a snapshot from whenever it was uploaded.
    if REPO_URL is not None:
        try:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                            REPO_URL, REPO_DIR], check=True, capture_output=True)
            source_used = f"git clone {REPO_URL} ({GITHUB_BRANCH})"
        except subprocess.CalledProcessError as error:
            print(f"git clone failed: {error.stderr.decode().strip()}")
    if not os.path.exists(REPO_DIR):
        attached = _find_attached_dataset()
        if attached is not None:
            shutil.copytree(attached, REPO_DIR)
            source_used = f"attached Kaggle dataset {attached}"
    if not os.path.exists(REPO_DIR):
        raise RuntimeError(
            "Could not obtain the stereo-depth source code.\n\n"
            f"git clone of {REPO_URL} failed and no attached Kaggle dataset contains "
            "train.py + stereo/model/stereo_net.py.\n\n"
            "  * Check the repository exists and is public, or\n"
            "  * Add Data -> Upload your local stereo-depth/ folder as a Kaggle Dataset "
            "and attach it.")
else:
    source_used = f"existing checkout (REFRESH_CODE={REFRESH_CODE})"

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Drop any modules imported from a previous, stale checkout in this kernel.
for module in [m for m in sys.modules if m == "stereo" or m.startswith("stereo.")]:
    del sys.modules[module]

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
print(f"source : {source_used}")
print(f"commit : {commit.stdout.strip() or 'unknown (not a git checkout)'}")

missing = _missing_modules(REPO_DIR)
if missing:
    raise RuntimeError(
        "This checkout is STALE -- it is missing modules this notebook needs:\n"
        + "".join(f"    {m}\n" for m in missing)
        + "\nThe code that reached Kaggle is older than the notebook. Fix whichever "
          "applies:\n"
          "  * pushed to GitHub?  Run 'git push' locally, then re-run this cell.\n"
          "  * using an attached Kaggle Dataset?  It is a snapshot -- upload a new "
          "version and attach that.\n"
          f"  * branch: this cloned {GITHUB_BRANCH!r}; set GITHUB_BRANCH if your work "
          "is elsewhere.")
print(f"modules: all {len(REQUIRED_MODULES)} required files present")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# Fast sanity check that the checkout is sound (~15 s).
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"], check=False)

## 4. Preflight — what your settings actually mean

Resolves every derived value, explains the consequences of your Control Panel choices, and
warns about inconsistencies **before** anything is downloaded or trained.

In [ ]:
from stereo.data.download import RECIPES
from stereo.geometry import compute_num_disparities

# How each dataset name maps to a training dataset spec. You do not edit this --
# it is driven by the DATASETS dict in the Control Panel.
DATASET_TYPES = {
    "middlebury": ("middlebury", {}),
    "eth3d":      ("eth3d",      {}),
    "kitti":      ("kitti",      {"reference_frames_only": False}),
    "kitti2015":  ("kitti",      {"version": "2015", "reference_frames_only": False}),
    "kitti2012":  ("kitti",      {"version": "2012", "reference_frames_only": False}),
    "sceneflow":  ("sceneflow",  {"split": SCENEFLOW_SPLIT}),
}
# Only pass options that were actually set, so None means "auto-detect".
DATASET_TYPES["sceneflow"][1].update(
    {k: v for k, v in (("subset", SCENEFLOW_SUBSET), ("pass_name", SCENEFLOW_PASS)) if v})
if KITTI_VERSION:
    DATASET_TYPES["kitti"][1]["version"] = KITTI_VERSION

# "kitti" here is raw/odometry recordings, which have no download recipe.
ATTACH_ONLY = {"kitti"}

# Options for the HDF5 loader, used when a mirror turns out to be a container
# rather than an image tree. The split keeps the benchmark keys out of training.
HDF5_OPTIONS = {"sceneflow": {"split": SCENEFLOW_SPLIT}}
PROTOCOL_TO_DATASET = {recipe.protocol: name for name, recipe in RECIPES.items()}

warnings = []

# --- resolve mode ------------------------------------------------------------
assert MODE in ("train_unlabeled", "adapt_unlabeled", "evaluate"), f"bad MODE: {MODE}"
OUTPUT_DIR = f"/kaggle/working/outputs/{MODE}"

# --- resolve datasets --------------------------------------------------------
ACTIVE_DATASETS = {name: weight for name, weight in DATASETS.items() if weight > 0}
for name in DATASETS:
    if name not in DATASET_TYPES:
        warnings.append(f"unknown dataset name {name!r}; known: {sorted(DATASET_TYPES)}")
    elif name in ATTACH_ONLY and name not in ATTACHED and DATASETS[name] > 0:
        warnings.append(f"{name!r} can only be attached, not downloaded. Add it to ATTACHED.")
if SCENEFLOW_SPLIT != "TRAIN":
    warnings.append(f"SCENEFLOW_SPLIT is {SCENEFLOW_SPLIT!r}. The paper's Table IV benchmark is "
                    "the Scene Flow TEST split; training on it would make that benchmark "
                    "meaningless. Use 'TRAIN'.")

def resolve_attached_path(configured):
    """Find an attached dataset even if the mount path differs from the configured one.

    Kaggle mounts datasets at /kaggle/input/<slug> in some setups and at
    /kaggle/input/datasets/<owner>/<slug> in others, so a hardcoded path is a
    coin flip. If the configured path is missing, the same final folder name is
    searched for under /kaggle/input.
    """
    if os.path.exists(configured):
        return configured
    leaf = os.path.basename(os.path.normpath(configured))
    for pattern in (f"/kaggle/input/{leaf}", f"/kaggle/input/*/{leaf}",
                    f"/kaggle/input/*/*/{leaf}", f"/kaggle/input/*/*/*/{leaf}"):
        for match in sorted(glob.glob(pattern)):
            if os.path.isdir(match):
                print(f"  resolved {configured} -> {match}")
                return match
    return configured


def dataset_root_for(name):
    """Attached Kaggle input wins over the download location."""
    if name in ATTACHED:
        return resolve_attached_path(ATTACHED[name])
    if name in ATTACH_ONLY:
        raise KeyError(f"{name!r} has no download recipe; attach it and list it in ATTACHED")
    return os.path.join(DATASET_ROOT, RECIPES[name].usage_root)

# Only non-attached datasets are ever downloaded.
TO_DOWNLOAD = [n for n in ACTIVE_DATASETS if n not in ATTACHED]

# --- restore a previous session ----------------------------------------------
RESTORED = {}
if RESUME_ARCHIVE and MODE != "evaluate":
    from stereo.utils.remote import restore_run
    try:
        RESTORED = restore_run(RESUME_ARCHIVE, OUTPUT_DIR)
        print(f"restored from the archive: {', '.join(sorted(RESTORED))}")
    except Exception as error:
        warnings.append(f"could not restore {RESUME_ARCHIVE}: {error}")

# --- resolve checkpoints -----------------------------------------------------
# A restored last.pt lets the trainer pick up optimiser, scheduler and history.
# epoch; best.pt would only supply weights, so it is a restart rather than a resume.
RESUME_CHECKPOINT = None
if RESTORED:
    candidate = RESTORED.get(f"{RESUME_WHICH}.pt")
    if candidate:
        RESUME_CHECKPOINT = candidate
    else:
        warnings.append(f"the archive has no {RESUME_WHICH}.pt "
                        f"(it has {', '.join(sorted(RESTORED))})")

RESOLVED_INIT = INIT_CHECKPOINT or "/kaggle/working/outputs/train_unlabeled/best.pt"
RESOLVED_CHECKPOINT = CHECKPOINT or f"{OUTPUT_DIR}/best.pt"
if MODE == "evaluate" and CHECKPOINT is None:
    # In evaluate mode there is no training run in this notebook to take best.pt from.
    RESOLVED_CHECKPOINT = "/kaggle/working/outputs/train_unlabeled/best.pt"

# --- resolve the evaluation root FROM the protocol (no second path to sync) ---
if PROTOCOL == "custom_folder":
    EVAL_ROOT = CUSTOM_DATA
elif PROTOCOL in PROTOCOL_TO_DATASET:
    EVAL_ROOT = dataset_root_for(PROTOCOL_TO_DATASET[PROTOCOL])
else:
    raise ValueError(f"unknown PROTOCOL {PROTOCOL!r}; known: "
                     f"{sorted(list(PROTOCOL_TO_DATASET) + ['custom_folder'])}")

# --- resolve the disparity range --------------------------------------------
# Fixed, NOT derived from TRAIN_WIDTH. The search range is baked into the
# weights, so deriving it from the input width means a different, incompatible
# network per resolution. CALIBRATED_DISPARITIES is set by cell 5b when
# DISPARITY_RANGE == "auto".
NUM_DISPARITIES = (DISPARITY_RANGE if isinstance(DISPARITY_RANGE, int)
                   else globals().get("CALIBRATED_DISPARITIES", 96))
MAX_DISPARITY_PX = (NUM_DISPARITIES // DOWNSAMPLE - 1) * DOWNSAMPLE - 1
cost_volume_mb = 16 * (NUM_DISPARITIES // DOWNSAMPLE) * (TRAIN_HEIGHT // DOWNSAMPLE) \
                 * (TRAIN_WIDTH // DOWNSAMPLE) * 4 / 1e6

# --- checks ------------------------------------------------------------------
if MODE == "train_unlabeled" and not ACTIVE_DATASETS:
    warnings.append("no dataset has a non-zero weight, so there is nothing to train on. "
                    "Set at least one entry in DATASETS above 0.")
if MODE == "adapt_unlabeled":
    if not os.path.isdir(CUSTOM_DATA):
        warnings.append(f"CUSTOM_DATA does not exist: {CUSTOM_DATA}. Attach your capture "
                        "as a Kaggle Dataset (a folder with left/ and right/).")
    if not os.path.isfile(RESOLVED_INIT):
        warnings.append(f"no checkpoint to adapt from at {RESOLVED_INIT}. Run "
                        "MODE='train_unlabeled' first, or set INIT_CHECKPOINT. Adaptation "
                        "from random weights is not what Stage 3 is for.")
if MODE == "evaluate":
    if not os.path.isfile(RESOLVED_CHECKPOINT):
        warnings.append(f"no checkpoint to evaluate at {RESOLVED_CHECKPOINT}. Train one "
                        "first or set CHECKPOINT.")
    needed = PROTOCOL_TO_DATASET.get(PROTOCOL)
    if needed and needed not in ACTIVE_DATASETS:
        warnings.append(f"PROTOCOL={PROTOCOL!r} needs the {needed!r} dataset, but its "
                        f"weight in DATASETS is 0, so it will not be downloaded. "
                        f"Set DATASETS[{needed!r}] above 0.")
if "eth3d" in ACTIVE_DATASETS and not shutil.which("7z"):
    warnings.append("eth3d needs 7z to extract. Run in a cell first: "
                    "!apt-get install -y -qq p7zip-full")
if "sceneflow" in ACTIVE_DATASETS and "sceneflow" not in ATTACHED:
    warnings.append("sceneflow is 132 GB and will exceed Kaggle's 20 GB working-dir quota. "
                    "Attach a Kaggle mirror instead and add it to ATTACHED, e.g. "
                    "ATTACHED = {'sceneflow': '/kaggle/input/sceneflow'}.")
if MODE == "evaluate" and PROTOCOL == "sceneflow" and "sceneflow" in ATTACHED:
    try:
        from stereo.data.sceneflow import discover_sceneflow as _discover
        if not _discover(ATTACHED["sceneflow"], "TEST", SCENEFLOW_PASS, SCENEFLOW_SUBSET).has_split:
            warnings.append(
                "the attached Scene Flow mirror has no TRAIN/TEST division, so its TEST split "
                "is the same data as TRAIN. Benchmarking it would score the model on its own "
                "training images, and the loader will refuse. Use PROTOCOL='middlebury2014', "
                "'eth3d' or 'kitti2015', which have real held-out splits.")
    except Exception:
        pass

for name, path in ATTACHED.items():
    if name not in DATASET_TYPES:
        warnings.append(f"ATTACHED has unknown dataset name {name!r}; "
                        f"known: {sorted(DATASET_TYPES)}")
    elif not os.path.isdir(path):
        warnings.append(f"ATTACHED[{name!r}] does not exist: {path}. Use Add Data to attach "
                        "it, then check the exact path under /kaggle/input.")
    elif DATASETS.get(name, 0) <= 0 and PROTOCOL_TO_DATASET.get(PROTOCOL) != name:
        warnings.append(f"{name!r} is attached but its weight in DATASETS is 0, so it will "
                        "not be used. Give it a non-zero weight.")
if BATCH_SIZE * cost_volume_mb > 11000:
    warnings.append(f"BATCH_SIZE={BATCH_SIZE} needs ~{BATCH_SIZE * cost_volume_mb / 1000:.1f} GB "
                    "for the cost volume alone, which will likely OOM on a 16 GB T4/P100. "
                    "Reduce BATCH_SIZE or TRAIN_WIDTH.")

# --- report ------------------------------------------------------------------
print("=" * 74)
print(f"MODE          : {MODE}")
if RESUME_CHECKPOINT:
    print(f"resuming      : {RESUME_CHECKPOINT}"
          + ("  (weights only -- optimiser state restarts)" if RESUME_WHICH == "best" else ""))
print(f"output        : {OUTPUT_DIR}")
if MODE == "train_unlabeled":
    described = ", ".join(f"{n} (weight {w}{', attached' if n in ATTACHED else ''})"
                          for n, w in ACTIVE_DATASETS.items())
    print(f"training on   : {described or 'NOTHING'}")
    print("                images only -- no ground truth is read in this mode")
    if MAX_TRAIN_SAMPLES:
        print(f"sample cap    : {MAX_TRAIN_SAMPLES} training pairs total, "
              f"split by weight")
    if TO_DOWNLOAD:
        print(f"will download : {', '.join(TO_DOWNLOAD)}")
    if ATTACHED:
        print(f"attached      : {', '.join(f'{n} -> {p}' for n, p in ATTACHED.items())}")
elif MODE == "adapt_unlabeled":
    print(f"adapting      : {RESOLVED_INIT}")
    print(f"           on : {CUSTOM_DATA} (images only)")
    print(f"learning rate : {LEARNING_RATE * ADAPT['lr_scale']:.1e} "
          f"(= LEARNING_RATE x {ADAPT['lr_scale']})")
else:
    print(f"evaluating    : {RESOLVED_CHECKPOINT}")
    print(f"protocol      : {PROTOCOL}")
    print(f"eval data     : {EVAL_ROOT}   <- derived from PROTOCOL")
    if MAX_EVAL_SAMPLES:
        print(f"sample cap    : first {MAX_EVAL_SAMPLES} images only")
    print("                GROUND TRUTH IS READ HERE, for metrics only")
print("-" * 74)
_geometry = ("height follows each image's aspect ratio" if PRESERVE_ASPECT
             else f"height {TRAIN_HEIGHT}")
print(f"resolution    : width {TRAIN_WIDTH} fixed, {_geometry}  (resized, never cropped)")
print(f"disparities   : {NUM_DISPARITIES}{'' if isinstance(DISPARITY_RANGE, int) else ' (provisional -- cell 5b measures it)'}"
      f" at a canonical width of {TRAIN_WIDTH}, "
      f"floored to a multiple of {DOWNSAMPLE}")
print(f"              : the model can see disparities of 0 .. {MAX_DISPARITY_PX} px")
print(f"              : i.e. objects no closer than  Z_min = f * B / {MAX_DISPARITY_PX}")
print(f"cost volume   : {cost_volume_mb:.0f} MB per image, "
      f"~{BATCH_SIZE * cost_volume_mb / 1000:.1f} GB at BATCH_SIZE={BATCH_SIZE}")
if MODE != "evaluate":
    print(f"objective     : {W_PHOTOMETRIC}*photo + {W_LEFT_RIGHT}*lr + {W_SMOOTHNESS}*smooth"
          f"{f' + {W_LOW_RESOLUTION}*lowres' if W_LOW_RESOLUTION else ''}"
          f"{f' + {W_CONFIDENCE}*conf' if W_CONFIDENCE else ''}")
    print("                the Monodepth objective; no ground-truth term exists")
print("=" * 74)

if warnings:
    print(f"\n{len(warnings)} WARNING(S):")
    for i, message in enumerate(warnings, 1):
        print(f"  {i}. {message}")
else:
    print("\nNo problems found.")

## 5. Datasets

Downloads exactly the datasets with a non-zero weight in `DATASETS`. Nothing else to edit.

**Training reads images only** — ground truth in these datasets is untouched until Stage 4.

In [ ]:
from stereo.data.download import describe, prepare, verify
print(describe())

In [ ]:
# Downloads whatever has a non-zero weight in DATASETS. In adapt mode your own
# capture is used instead, so nothing is downloaded unless you also want to evaluate.
to_fetch = list(ACTIVE_DATASETS)
if MODE == "evaluate":
    needed = PROTOCOL_TO_DATASET.get(PROTOCOL)
    if needed and needed not in to_fetch:
        to_fetch.append(needed)
if MODE == "adapt_unlabeled":
    to_fetch = [n for n in to_fetch if n == PROTOCOL_TO_DATASET.get(PROTOCOL)]

# Anything in ATTACHED is already on disk as a Kaggle input -- never download it.
skipped = [n for n in to_fetch if n in ATTACHED or n in ATTACH_ONLY]
to_fetch = [n for n in to_fetch if n not in ATTACHED and n not in ATTACH_ONLY]
for name in skipped:
    print(f"{name}: using attached input {ATTACHED.get(name, '(attach-only)')} (not downloading)")

print(f"fetching: {to_fetch or 'nothing (using your own data)'}\n")
for name in to_fetch:
    try:
        prepare(name, DATASET_ROOT)
    except Exception as error:
        print(f"{name} FAILED: {error}")

verify(DATASET_ROOT)

### Attached datasets — what was actually found

Each loader searches its attached directory for the pair of views, **at any nesting depth**,
rather than assuming a path — Kaggle mirrors reshape public datasets freely. This cell builds
each dataset exactly as training will and reports what it resolved, so an unrecognised mirror
shows you its real directory tree instead of failing opaquely.

| Dataset | Layouts handled |
|---|---|
| FlyingThings3D | `frames_finalpass/TRAIN/<A\|B\|C>/<scene>/left`, the flat `subset` release, or bare `left`/`right` trees with no pass directory |
| KITTI | raw / Eigen (`<date>/<drive>/image_02/data`), 2015 (`image_2`), 2012 (`colored_0`) |
| Middlebury | any folder containing `im0.png` + `im1.png` |

If something is wrong, paste this cell's output back and the layout can be added.

In [ ]:
from stereo.data import build_dataset, DatasetMode
from stereo.data.discovery import describe_tree
from stereo.data.hdf5_stereo import find_hdf5_files, inspect_hdf5
from stereo.data.registry import DatasetSpec

USABLE_DATASETS = {}      # name -> weight, for the datasets that actually loaded
DATASET_ERRORS = {}       # name -> message, for the ones that did not
TYPE_OVERRIDES = {}       # name -> loader type, when the mirror is not the expected format

for name, weight in ACTIVE_DATASETS.items():
    path = dataset_root_for(name)
    print("=" * 74)
    print(f"{name}  (weight {weight})  ->  {path}")
    print("=" * 74)

    if not os.path.isdir(path) and not os.path.isfile(path):
        DATASET_ERRORS[name] = f"path does not exist: {path}"
        print("  MISSING. Use Add Data, then copy the exact path from /kaggle/input:")
        for entry in sorted(glob.glob("/kaggle/input/*")):
            print("   ", entry)
        continue

    # A container-style mirror ships one .hdf5 instead of a tree of images.
    containers = find_hdf5_files(path)
    if containers:
        print("  This mirror is an HDF5 container, not an image tree.")
        print("  " + inspect_hdf5(path).replace("\n", "\n  "))
        print()

    dataset_type, options = DATASET_TYPES[name]
    try:
        dataset = build_dataset(DatasetSpec(type=dataset_type, root=path, options=options),
                                DatasetMode.TRAIN, None)
        sample = dataset[0]
    except Exception as error:
        # A mirror shipped as an HDF5 container will not load as an image tree.
        # If it turns out to hold stereo pairs, use it; if not, the HDF5 loader
        # says so explicitly and we fall back to reporting the original failure.
        if containers:
            try:
                dataset = build_dataset(
                    DatasetSpec(type="hdf5", root=path,
                                options=HDF5_OPTIONS.get(name, {})),
                    DatasetMode.TRAIN, None)
                sample = dataset[0]
                TYPE_OVERRIDES[name] = "hdf5"
                print("  loaded via the HDF5 loader instead of the image-tree loader")
            except Exception as hdf5_error:
                DATASET_ERRORS[name] = f"{type(hdf5_error).__name__}: {hdf5_error}"
                print(f"  COULD NOT LOAD as an image tree or as an HDF5 stereo container.")
                print("  " + str(hdf5_error).replace("\n", "\n  "))
                continue
        else:
            DATASET_ERRORS[name] = f"{type(error).__name__}: {error}"
            print(f"  COULD NOT LOAD: {type(error).__name__}")
            print("  " + str(error).replace("\n", "\n  "))
            continue

    assert not any(k in sample for k in ("disparity_gt", "depth_gt", "valid_gt_mask"))
    USABLE_DATASETS[name] = weight

    # Stereo searches horizontal scanlines only, so a vertical offset between the
    # views means the correct match is not on the line being searched -- no amount
    # of training recovers from that. Worth checking on data that has been
    # augmented, cropped or re-calibrated.
    from stereo.data.discovery import check_rectification
    offsets = [check_rectification(dataset[i]["left"], dataset[i]["right"])["vertical_offset"]
               for i in range(min(3, len(dataset)))]
    if any(o != 0 for o in offsets):
        print(f"  WARNING: vertical offsets {offsets} between the views. A rectified pair")
        print("           must have none; stereo matching searches horizontal lines only.")
    else:
        print(f"  rectified    : yes (checked {len(offsets)} pairs)")
    print(f"  OK: {len(dataset)} stereo pairs")
    print(f"  first sample : {sample['metadata']['sample_id']}  {tuple(sample['left'].shape)}")
    print("  confirmed    : no ground truth in the training sample")
    if hasattr(dataset, "layout"):
        print("  " + dataset.layout.describe().replace("\n", "\n  "))
        if not dataset.layout.has_split:
            print("  NOTE: no TRAIN/TEST division -- fine for training, but this mirror")
            print("        cannot be benchmarked (its TEST split is its TRAIN split).")
    if hasattr(dataset, "version"):
        print(f"  kitti layout : {dataset.version}")
        print(f"  calibration  : "
              f"{'found' if sample['metadata'].get('focal_length') else 'NOT found'}")
    if hasattr(dataset, "describe") and not hasattr(dataset, "layout"):
        print("  " + dataset.describe().replace("\n", "\n  "))

def sample_cap(name, weight):
    """This dataset's share of MAX_TRAIN_SAMPLES, in proportion to its weight."""
    if not MAX_TRAIN_SAMPLES:
        return None
    total = sum(w for n, w in USABLE_DATASETS.items()) or 1.0
    return max(1, int(round(MAX_TRAIN_SAMPLES * weight / total)))


def spec_for(name, weight=1.0):
    """The dataset spec that training will actually use.

    Defined here, once, because the diagnostic above is what discovers that a
    mirror needs a different loader than its name implies (an HDF5 container
    rather than an image tree). Every later cell goes through this, so the
    preview and the training mixture cannot drift apart.
    """
    cap = sample_cap(name, weight)
    if name in TYPE_OVERRIDES:
        return DatasetSpec(type=TYPE_OVERRIDES[name], root=dataset_root_for(name),
                           weight=weight, options=HDF5_OPTIONS.get(name, {}), max_samples=cap)
    dataset_type, options = DATASET_TYPES[name]
    return DatasetSpec(type=dataset_type, root=dataset_root_for(name),
                       weight=weight, options=options, max_samples=cap)


print("=" * 74)
print(f"usable for training : {', '.join(USABLE_DATASETS) or 'NONE'}")
if DATASET_ERRORS:
    print(f"unusable            : {', '.join(DATASET_ERRORS)}")
    print("\nTraining will proceed with the usable ones only. Fix or remove the others,")
    print("or paste the output above if a layout needs to be added.")
if not USABLE_DATASETS:
    raise RuntimeError("no attached dataset could be loaded; see the errors above")


## 5b. Measure the disparity range — from images, no labels

The search range is part of the weights, so it is chosen here, once, before the model
exists. `DISPARITY_RANGE = "auto"` block-matches your own training pairs against each
other and reports the range they need; an integer pins it by hand.

No ground truth is opened — this runs on the training split.


In [ ]:
# ---------------------------------------------------------------------------
#  How far does this data actually need to search?
# ---------------------------------------------------------------------------
# The search range is part of the weights, so it must be chosen before training,
# and both directions cost: too small clips the near field, too large loses
# accuracy because every extra candidate is another chance at a spurious match.
# Measured on a real Middlebury pair at 640x384 (block-matching MAE, native px):
#
#     range   48    64    96   128   192   256   320
#       MAE 6.42  6.23  6.97  7.44  8.48  9.73 10.83
#
# So min(width // 2, 384) -- which gives 320 here -- is close to the worst
# available choice. This measures the range your data needs instead.
#
# LABEL-FREE: block-matches the left and right views against each other and takes
# a percentile of the matches that pass a ratio test. No ground truth is opened.
from stereo.data import DatasetMode, build_dataset, calibrate_disparity_range
from stereo.data.augmentation import ResizeConfig, build_train_transform
from stereo.data.registry import DatasetSpec, build_loader

if isinstance(DISPARITY_RANGE, int):
    CALIBRATED_DISPARITIES = DISPARITY_RANGE
    print(f"DISPARITY_RANGE pinned to {DISPARITY_RANGE} px at {TRAIN_WIDTH} wide "
          f"(set it to \"auto\" to measure it instead)")
else:
    calib_transform = build_train_transform(ResizeConfig(height=TRAIN_HEIGHT, width=TRAIN_WIDTH,
                                 preserve_aspect=PRESERVE_ASPECT), None)
    names = ["__custom__"] if MODE == "adapt_unlabeled" else list(USABLE_DATASETS)
    per_dataset = max(CALIB_PAIRS // max(len(names), 1), 1)

    batches = []
    for name in names:
        spec = (DatasetSpec(type="folder", root=CUSTOM_DATA) if name == "__custom__"
                else spec_for(name))
        dataset = build_dataset(spec, DatasetMode.TRAIN, calib_transform)
        loader = build_loader(dataset, batch_size=1, shuffle=True, num_workers=0)
        for index, batch in enumerate(loader):
            if index >= per_dataset:
                break
            batches.append({"left": batch["left"], "right": batch["right"]})

    estimate = calibrate_disparity_range(
        batches, canonical_width=TRAIN_WIDTH, percentile=CALIB_PERCENTILE,
        margin=CALIB_MARGIN, downsample=DOWNSAMPLE, max_pairs=CALIB_PAIRS)
    print(estimate)
    if estimate.saturated:
        print("\n  The estimate hit the search ceiling, so the true range may be larger.")
        print("  Re-run with a larger search_fraction, or pin DISPARITY_RANGE by hand.")
    CALIBRATED_DISPARITIES = estimate.recommended

MAX_DISPARITY_PX = (CALIBRATED_DISPARITIES // DOWNSAMPLE - 1) * DOWNSAMPLE - 1
print(f"\n-> num_disparities = {CALIBRATED_DISPARITIES}; the model will see "
      f"0 .. {MAX_DISPARITY_PX} px at {TRAIN_WIDTH} wide")
print(f"   = {CALIBRATED_DISPARITIES / TRAIN_WIDTH:.1%} of image width, which is what "
      f"carries over to other resolutions")


## 6. Look at the training pairs

Images only — no ground truth is loaded here, in any mode. The assertion below proves it.

In [ ]:
import matplotlib.pyplot as plt
from stereo.data import DatasetMode, build_dataset
from stereo.data.augmentation import PhotometricAugmentConfig, ResizeConfig, build_train_transform
from stereo.data.registry import DatasetSpec

transform = build_train_transform(ResizeConfig(height=TRAIN_HEIGHT, width=TRAIN_WIDTH,
                                 preserve_aspect=PRESERVE_ASPECT),
                                  PhotometricAugmentConfig(enabled=True, probability=1.0), seed=0)

# Preview whatever the Control Panel actually selected -- no hard-coded dataset.
if MODE == "adapt_unlabeled":
    preview_spec = DatasetSpec(type="folder", root=CUSTOM_DATA)
else:
    # Preview one of the datasets that actually loaded, not merely one that was configured.
    first = next(iter(USABLE_DATASETS), None)
    if first is None:
        raise RuntimeError("no usable dataset; see the diagnostic cell above")
    preview_spec = spec_for(first)

preview = build_dataset(preview_spec, DatasetMode.TRAIN, transform)
print(f"{preview_spec.type} @ {preview_spec.root}")
print(f"{len(preview)} pairs | sample keys: {sorted(preview[0])}")
assert not any(k in preview[0] for k in ("disparity_gt", "depth_gt", "valid_gt_mask"))
print("confirmed: the training sample carries no ground truth")

rows = min(3, len(preview))
fig, axes = plt.subplots(rows, 2, figsize=(13, 3 * rows), squeeze=False)
for row in range(rows):
    sample = preview[row]
    for col, view in enumerate(("left", "right")):
        axes[row][col].imshow(sample[view].permute(1, 2, 0).numpy())
        axes[row][col].set_title(f"{view} (augmented) - {sample['metadata']['sample_id']}")
        axes[row][col].axis("off")
plt.tight_layout(); plt.show()

## 7. Assemble the training configuration

Pure translation of the Control Panel into the repo's `Config` object. Nothing new to decide
here — it is printed so you can see exactly what was built, and saved next to the checkpoints
so the run is reproducible from the CLI.

In [ ]:
from stereo.config import Config, LossWeights, config_to_yaml
from stereo.data.augmentation import GeometricAugmentConfig
from stereo.model import StereoNetConfig

config = Config()
config.dynamic_disparity = False     # already resolved in the preflight
# num_disparities is FIXED, not derived from TRAIN_WIDTH: it is part of the
# weights, so a width-derived range would mean a different, incompatible network
# at every resolution. canonical_width records the width it is expressed at, so
# inference at any other size can rescale (see cell 8).
config.model = StereoNetConfig(num_disparities=CALIBRATED_DISPARITIES,
                               downsample=DOWNSAMPLE,
                               canonical_width=TRAIN_WIDTH,
                               soft_argmin_window=SOFT_ARGMIN_WINDOW,
                               residual_limit_fraction=RESIDUAL_LIMIT)

config.data.resize = ResizeConfig(height=TRAIN_HEIGHT, width=TRAIN_WIDTH,
                                 preserve_aspect=PRESERVE_ASPECT)
config.data.photometric_augmentation = PhotometricAugmentConfig(enabled=True)
config.data.geometric_augmentation = GeometricAugmentConfig(enabled=True, scale=(0.8, 1.2),
                                                            aspect=(0.9, 1.1))

# Training mixture, straight from the DATASETS dict.
if MODE == "adapt_unlabeled":
    config.data.train = [DatasetSpec(type="folder", root=CUSTOM_DATA, weight=1.0)]
    config.training.init_checkpoint = RESOLVED_INIT
    config.optimizer.learning_rate = LEARNING_RATE * ADAPT["lr_scale"]
else:
    # Only datasets the diagnostic cell actually loaded.
    config.data.train = [spec_for(name, weight) for name, weight in USABLE_DATASETS.items()]
    if DATASET_ERRORS:
        print(f"NOTE: excluded {', '.join(DATASET_ERRORS)} -- see the diagnostic cell")
    config.optimizer.learning_rate = LEARNING_RATE

config.data.validation = list(config.data.train)      # label-free validation

config.loss = LossWeights(photometric=W_PHOTOMETRIC, left_right=W_LEFT_RIGHT,
                          smoothness=W_SMOOTHNESS, low_resolution=W_LOW_RESOLUTION,
                          confidence=W_CONFIDENCE, range_penalty=W_RANGE_PENALTY)

config.training.epochs = EPOCHS
config.training.batch_size = BATCH_SIZE
config.training.num_workers = NUM_WORKERS
config.training.use_amp = torch.cuda.is_available()
config.training.output_dir = OUTPUT_DIR
config.training.selection_metric = "val/photometric"   # LABEL-FREE checkpoint selection
if RESUME_CHECKPOINT:
    config.training.resume = RESUME_CHECKPOINT
config.training.visualize_every = VISUALIZE_EVERY
config.training.visualize_samples = VISUALIZE_SAMPLES

os.makedirs(OUTPUT_DIR, exist_ok=True)
open(f"{OUTPUT_DIR}/config.yaml", "w").write(config_to_yaml(config))

print(f"num_disparities  : {config.model.num_disparities}")
print(f"training sets    : {[(s.type, s.weight) for s in config.data.train]}")
print(f"init checkpoint  : {config.training.init_checkpoint or 'none (random init)'}")
print(f"learning rate    : {config.optimizer.learning_rate:.1e}")
print(f"saved config     : {OUTPUT_DIR}/config.yaml")

## 8. Build the model

Random initialisation — no ImageNet weights, no pretrained stereo weights, no pretrained
Monodepth weights.

In [ ]:
from stereo.model import StereoNet

model = StereoNet(config.model)
print(f"parameters      : {model.num_parameters():,}")
print(f"cost volume     : {model.num_disparities_small} levels at 1/{model.scale}")
print(f"search range    : 0 .. {model.max_disparity} px at full resolution")
print(f"size divisor    : {model.size_divisor} (inputs are padded right/bottom, then cropped back)")

# Arbitrary resolution, including sizes that need padding.
model.eval()
for height, width in [(TRAIN_HEIGHT, TRAIN_WIDTH), (375, 1242), (540, 960), (224, 224)]:
    with torch.no_grad():
        out = model.forward_left(torch.rand(1, 3, height, width), torch.rand(1, 3, height, width))
    print(f"  {width}x{height} -> disparity {tuple(out['disparity'].shape)}")

## 9. Train, label-free

Stage 1 is photometric + smoothness + left-right consistency from random init.
Stage 2 adds the EMA teacher at `
**Watch `pseudo_cov`** once the teacher starts: near 0 means the reliability filter is
rejecting everything (loosen `FILTER_*`), near 1 means it is copying the teacher's errors
wholesale (tighten them). The trainer warns in both cases.

In [ ]:
if MODE == "evaluate":
    print("MODE is 'evaluate'; skipping training.")
else:
    from stereo.training import Trainer

    trainer = Trainer(config)
    best_checkpoint = trainer.fit()
    print("best label-free checkpoint:", best_checkpoint)

## 10. Label-free validation curves

No ground-truth metric is plotted — these are the quantities that actually selected the
checkpoint.

In [ ]:
import json

history_path = f"{OUTPUT_DIR}/history.json"
if os.path.exists(history_path):
    history = json.load(open(history_path))
    panels = [("train/total", "total loss"), ("train/photometric", "photometric"),
              ("train/left_right", "left-right consistency"), ("train/smoothness", "smoothness"),
              ("train/mean_confidence", "mean confidence")]
    fig, axes = plt.subplots(2, 3, figsize=(16, 7))
    for axis, (key, title) in zip(axes.flat, panels):
        values = [record.get(key) for record in history]
        if any(v is not None for v in values):
            axis.plot([r["epoch"] for r in history], values, marker="o", ms=3)
        if key == "train/photometric" and "val/photometric" in history[0]:
            axis.plot([r["epoch"] for r in history], [r["val/photometric"] for r in history],
                      marker="s", ms=3, label="validation")
            axis.legend()
        axis.set_title(title); axis.set_xlabel("epoch"); axis.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("no history yet")

## 11. Qualitative check — still no ground truth

Predicted disparity, confidence, and the photometric reconstruction the model was actually
trained on.

In [ ]:
from stereo.data import collate_samples
from stereo.geometry import warp_right_to_left
from stereo.utils.visualization import colorize, to_numpy_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

batch = collate_samples([preview[i] for i in range(min(2, len(preview)))])
left, right = batch["left_clean"].to(device), batch["right_clean"].to(device)
with torch.no_grad():
    outputs = model(left, right, directions=("left", "right"))

reconstruction, valid = warp_right_to_left(right, outputs["left"]["disparity"])
residual = (reconstruction - left).abs().mean(dim=1, keepdim=True) * valid

rows = left.shape[0]
fig, axes = plt.subplots(rows, 3, figsize=(16, 3.5 * rows), squeeze=False)
for row in range(rows):
    for axis, (image, title) in zip(axes[row], [
            (to_numpy_image(left[row:row + 1]), "left"),
            (colorize(outputs["left"]["disparity"][row]), "predicted disparity"),
            (colorize(outputs["left"]["confidence"][row], 0, 1, cmap="viridis"),
             "confidence = exp(matchability)")]):
        axis.imshow(image); axis.set_title(title); axis.axis("off")
plt.tight_layout(); plt.show()
print(f"photometric residual (the training signal): {float(residual.sum() / valid.sum()):.4f}")

---
# 12. GROUND-TRUTH EVALUATION

**Everything below reads ground truth and runs only when `MODE == "evaluate"`.**

The checkpoint is loaded, frozen, and measured. No optimiser is constructed, and the
checkpoint being measured was selected by a label-free criterion, so ground truth did not
influence which weights got here either.

In [ ]:
if MODE != "evaluate":
    print(f"MODE is '{MODE}'. Ground truth is NOT loaded.")
    print("Set MODE = 'evaluate' in the Control Panel and re-run from the top.")
else:
    from stereo.evaluation import (PUBLISHED_RESULTS, evaluate_checkpoint, format_summary,
                                   get_protocol, write_results)
    from stereo.postprocess import PostProcessConfig
    from stereo.utils.checkpoint import build_model_from_checkpoint, checkpoint_hash

    frozen = build_model_from_checkpoint(RESOLVED_CHECKPOINT, map_location=device)
    protocol = get_protocol(PROTOCOL)
    print(f"checkpoint : {RESOLVED_CHECKPOINT}")
    print(f"sha256     : {checkpoint_hash(RESOLVED_CHECKPOINT)[:16]}...")
    print(f"protocol   : {protocol.name}")
    print(f"data       : {EVAL_ROOT}")
    print(f"notes      : {protocol.notes}")

### 12a. The paper's protocol: raw output

The paper's tables use *only the raw output of the learned model*, so post-processing is off.

In [ ]:
if MODE == "evaluate":
    summary_raw = evaluate_checkpoint(frozen, protocol, EVAL_ROOT, device,
                                      PostProcessConfig(enabled=False),
                                      max_samples=MAX_EVAL_SAMPLES)
    summary_raw["checkpoint"] = RESOLVED_CHECKPOINT
    write_results(summary_raw, f"{OUTPUT_DIR}/evaluation/{PROTOCOL}/raw")
    print(format_summary(summary_raw))

### 12b. With matchability post-processing

`exp(matchability) >= POSTPROCESS_CONFIDENCE` and a minimum depth-region of
`POSTPROCESS_MIN_REGION` px, as in the paper's on-robot pipeline. Reported separately, with
the pixel coverage stated, so the effect of the model and the effect of the filter stay
distinguishable.

In [ ]:
if MODE == "evaluate":
    summary_pp = evaluate_checkpoint(
        frozen, protocol, EVAL_ROOT, device,
        PostProcessConfig(enabled=True, confidence_threshold=POSTPROCESS_CONFIDENCE,
                          min_region_pixels=POSTPROCESS_MIN_REGION),
        max_samples=MAX_EVAL_SAMPLES)
    summary_pp["checkpoint"] = RESOLVED_CHECKPOINT
    write_results(summary_pp, f"{OUTPUT_DIR}/evaluation/{PROTOCOL}/postprocessed")
    print(format_summary(summary_pp))

### 12c. Comparison table

Every row is labelled *published*, *measured* or *not available*. Nothing is invented, and
rows measured under different protocols are never presented as equivalent.

In [ ]:
if MODE == "evaluate":
    def get(summary, key, variant="all"):
        # .get on the variant too: a protocol without a nonocc mask has no "nonocc" row.
        value = summary["disparity_metrics"].get(variant, {}).get(key)
        return f"{value:.3f}" if isinstance(value, (int, float)) else "n/a"

    rows = []
    if PROTOCOL == "sceneflow":
        published = PUBLISHED_RESULTS["sceneflow"]["metrics"]
        rows.append(("Original paper (Table IV)", "supervised, GT disparity",
                     f"{published['global_epe']:.3f}", f"{published['global_bad_1']:.1f}",
                     "PUBLISHED"))
        rows.append(("Reference mmstereo checkpoint", "supervised", "n/a", "n/a",
                     "NOT AVAILABLE (no released weights)"))
        rows.append(("This implementation (raw)", "label-free self-supervised",
                     get(summary_raw, "global_epe"), get(summary_raw, "global_bad_1"), "MEASURED"))
        rows.append(("This implementation (post-processed)", "label-free self-supervised",
                     get(summary_pp, "global_epe"), get(summary_pp, "global_bad_1"),
                     f"MEASURED on {summary_pp['ground_truth_pixel_coverage'] * 100:.0f}% of pixels"))
        header = ("Model", "Training", "EPE (px)", "%Bad(1.0)", "Status")
    elif PROTOCOL == "middlebury2014":
        published = PUBLISHED_RESULTS["middlebury2014_test"]["metrics"]
        rows.append(("Original paper (Table V, TEST split)", "supervised, GT disparity",
                     f"{published['image_bad_2_nonocc']}/{published['image_bad_2_all']}",
                     f"{published['image_avgerr_nonocc']}/{published['image_avgerr_all']}",
                     "PUBLISHED - DIFFERENT SPLIT"))
        rows.append(("This implementation (raw, TRAINING split)", "label-free self-supervised",
                     f"{get(summary_raw, 'image_bad_2', 'nonocc')}/{get(summary_raw, 'image_bad_2')}",
                     f"{get(summary_raw, 'image_avgerr', 'nonocc')}/{get(summary_raw, 'image_avgerr')}",
                     "MEASURED"))
        header = ("Model", "Training", "bad2.0 nocc/all", "avgerr nocc/all", "Status")
    else:
        rows.append((f"This implementation (raw)", "label-free self-supervised",
                     get(summary_raw, "global_epe"), get(summary_raw, "global_bad_1"), "MEASURED"))
        rows.append((f"This implementation (post-processed)", "label-free self-supervised",
                     get(summary_pp, "global_epe"), get(summary_pp, "global_bad_1"), "MEASURED"))
        header = ("Model", "Training", "EPE (px)", "%Bad(1.0)", "Status")
        print(f"NOTE: the paper reports no accuracy numbers for {PROTOCOL}, so there is no "
              "published row to compare against.\n")

    widths = [max(len(str(row[i])) for row in [header] + rows) for i in range(len(header))]
    line = lambda row: "  ".join(str(cell).ljust(widths[i]) for i, cell in enumerate(row))
    print(f"Dataset  : {protocol.dataset_type}   Split: {protocol.split}")
    print(f"Protocol : {protocol.primary_source}")
    print(f"Mask     : {protocol.max_disparity_source}")
    print(f"Scaling  : {'median' if protocol.median_scaling else 'none (metric prediction)'}\n")
    print(line(header)); print("-" * (sum(widths) + 2 * len(widths)))
    for row in rows: print(line(row))
    if PROTOCOL == "middlebury2014":
        print(f"\nCAVEAT: {PUBLISHED_RESULTS['middlebury2014_test']['caveat']}")

### 12d. Error visualisation

Sparse ground truth is never densified for display — invalid pixels stay grey.

In [ ]:
if MODE == "evaluate":
    from stereo.data import build_benchmark_dataset
    from stereo.evaluation.disparity_metrics import disparity_valid_mask
    from stereo.postprocess import upsample_confidence

    dataset = build_benchmark_dataset(DatasetSpec(type=protocol.dataset_type, root=EVAL_ROOT,
                                                  options=dict(protocol.dataset_options)))
    for index in range(min(2, len(dataset))):
        sample = collate_samples([dataset[index]])
        left_image, right_image = sample["left"].to(device), sample["right"].to(device)
        with torch.no_grad():
            output = frozen.forward_left(left_image, right_image)

        disparity = output["disparity"]
        disparity_gt = sample["disparity_gt"].to(device)
        valid = disparity_valid_mask(disparity_gt, protocol.max_disparity, protocol.min_disparity,
                                     sample["valid_gt_mask"].to(device))
        error = (disparity - disparity_gt).abs()
        vmax = float(disparity_gt[valid].max()) if valid.any() else None

        panels = [(to_numpy_image(left_image), "left image"),
                  (colorize(disparity, 0, vmax), "predicted disparity"),
                  (colorize(disparity_gt, 0, vmax, mask=valid), "ground-truth disparity"),
                  (colorize(error, 0, 5, mask=valid, cmap="inferno"), "|error| (px), 0-5"),
                  (colorize(valid.float(), 0, 1, cmap="gray"), "valid GT mask"),
                  (colorize(upsample_confidence(output["confidence"], disparity.shape[-2:]), 0, 1,
                            cmap="viridis"), "confidence")]
        fig, axes = plt.subplots(2, 3, figsize=(17, 7))
        for axis, (image, title) in zip(axes.flat, panels):
            axis.imshow(image); axis.set_title(title, fontsize=10); axis.axis("off")
        epe = float(error[valid].mean()) if valid.any() else float("nan")
        fig.suptitle(f"{sample['metadata']['sample_id'][0]} - EPE {epe:.3f} px")
        plt.tight_layout(); plt.show()

## 13. Label-leakage audit

Runs in every mode. If this fails, nothing else in the notebook means anything.

In [ ]:
subprocess.run([sys.executable, "scripts/audit_label_leakage.py", "--strict"], check=True)

## 14. Save artefacts

Kaggle keeps `/kaggle/working`. Checkpoints and evaluation output are already there; this
just lists what a rerun would pick up.

In [ ]:
from stereo.utils.remote import package_run

# Pack this session's checkpoints so the next one can pick up where this stopped.
archive = package_run(OUTPUT_DIR)
if archive:
    print(f"resume archive: {archive}  ({os.path.getsize(archive) / 1e6:.1f} MB)")
    print("\nTo continue in a later session:")
    print("  1. download it from the Kaggle output panel (or /kaggle/working)")
    print("  2. upload it to Google Drive and share it 'Anyone with the link'")
    print("  3. paste that link into RESUME_ARCHIVE in the Control Panel")
    print("  (an archive attached as a Kaggle Dataset works too -- give its path)")
else:
    print(f"nothing to package yet under {OUTPUT_DIR}")

print("\nEverything under /kaggle/working/outputs:")
for directory, _, filenames in os.walk("/kaggle/working/outputs"):
    for filename in sorted(filenames):
        path = os.path.join(directory, filename)
        print(f"{os.path.getsize(path) / 1e6:8.2f} MB  {path}")
